# Kaggle bootstrap — hotel data analysis

This notebook wires the repo into a Kaggle session: clone the project, confirm **raw** CSV paths, then use a **joined** Parquet (processed hotels + world cities) for modeling.

**Suggested order**

1. Attach **raw** datasets (hotels, world cities, crime) via **Add data** — needed for path checks and optional previews.
2. **Clone** the repository into `/kaggle/working/project`.
3. **Verify** that each raw CSV is found under `/kaggle/input` (cell below prints paths).
4. *(Optional)* Run **head/peek** scripts to sanity-check encodings.
5. Attach your **processed** joined Parquet (`hotels_with_cities*.parquet`) as a separate dataset, or generate it (see the **reference-only** cleaning command — do not run if you already have the file).
6. **Show** the joined Parquet file path, then **train** the baseline model.

Constants used below: `PROJECT = /kaggle/working/project` (clone target). 

## Environment

Kaggle kernels already ship with **pandas** and **pyarrow** (enough for Parquet I/O). You do **not** need `!pip install pandas pyarrow` unless you require a specific newer version.

**scikit-learn** is installed in the training section; install it there only if `import sklearn` fails.

## 1. Raw CSV datasets (prerequisite)

In the Kaggle notebook sidebar, use **Add data** and attach:

- Hotels CSV  
- World cities (`worldcitiespop.csv` or equivalent)  
- Crime index CSV *(optional)*  


## 2. Clone the repository

Clones into `/kaggle/working/project` so `import src...` works after `sys.path` is set. Edit `REPO_URL` to match your team’s fork or branch.

In [ ]:
REPO_URL = "https://github.com/daniahsn/hotel-data-analysis.git"

import shutil
from pathlib import Path

dest = Path("/kaggle/working/project")
if dest.exists():
    shutil.rmtree(dest)
!git clone {REPO_URL} {dest}

## 3. Verify raw CSV locations

Prints the path to each raw file (`hotels`, `world`, `crime`). If anything is missing, re-check **Add data** and dataset names under `/kaggle/input`.

In [ ]:
import sys
from pathlib import Path

PROJECT = Path("/kaggle/working/project")
sys.path.insert(0, str(PROJECT))

from src.raw_data_paths import DATASET_ORDER, raw_csv_paths, is_kaggle_runtime

paths = raw_csv_paths()
print("runtime:", "Kaggle (/kaggle/input)" if is_kaggle_runtime() else "local (data/raw)")
for key in DATASET_ORDER:
    print(key, "->", paths[key])

## 4. Processed joined Parquet

Modeling expects a file named like **`hotels_with_cities.parquet`** (or `hotels_with_cities*.parquet`).

- **Typical workflow:** run the long cleaning job once, upload the result as a **Kaggle Dataset**, then **Add data** here so it appears under `/kaggle/input`.
- Training looks for **`hotels_with_cities*.parquet`** under `/kaggle/input` (Kaggle) or `data/processed/` (local). The next code cell prints the exact file path.

The next cell is **documentation only**: the command used to **produce** that Parquet from raw CSVs. **Do not run** it in this notebook if you already attached the output — full runs can take many hours.

### Reference only — command that created `hotels_with_cities*.parquet`

**Do not execute** unless you intend to regenerate processed data from scratch on Kaggle.

```bash
# From repo root after clone; writes to /kaggle/working/processed (download or re-upload as a Dataset).
cd /kaggle/working/project
pip install -q -r requirements.txt

# Full-scale run (chunked — adjust chunksize / progress; runtime is long):
python scripts/pipeline/run_cleaning.py \
  --output-dir /kaggle/working/processed \
  --chunksize 500000 \
  --progress-every 2000000 \
  --format parquet

# Quick smoke test only (small sample, minutes):
# python scripts/pipeline/run_cleaning.py --output-dir /kaggle/working/processed --sample 10000 --format parquet
```

Outputs include `hotels_clean`, `world_cities_clean`, and **`hotels_with_cities`** (the joined table used for features).

## 6. Joined Parquet file path

Prints which file `load_joined_hotels()` and training will use. If this errors, attach your processed dataset or check filenames match `hotels_with_cities*.parquet`.

In [ ]:
import sys
from pathlib import Path

PROJECT = Path("/kaggle/working/project")
sys.path.insert(0, str(PROJECT))

from src.raw_data_paths import joined_hotels_parquet_path

joined = joined_hotels_parquet_path(PROJECT)
print("joined Parquet:", joined)

## 7. Train baseline regressor

Installs **scikit-learn** if needed, then runs `scripts/modeling/train_baseline_model.py`.

- Use **`--sample`** while iterating (faster, less RAM). Remove `--sample` for a full-data run when you are ready.
- Artifacts go to **`/kaggle/working/model_artifacts/`** by default (`regressor.joblib`, `preprocessor.joblib`, `metrics.json`).

In [ ]:
%cd /kaggle/working/project
!pip install -q scikit-learn

# While testing, keep --sample; for full data, remove --sample (heavy).
!python scripts/modeling/train_baseline_model.py --sample 100000 --model rf --project-root /kaggle/working/project